# Notebook 5: Final Results and Paper Figures

**Goal:** Generate all tables and figures used in the paper.

Run this notebook LAST after all other notebooks are complete.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('../')

from src.model import CNN
from src.attacks import fgsm_attack, pgd_attack
from src.ga import GeneticAttack

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

baseline_model = CNN().to(device)
baseline_model.load_state_dict(torch.load('../results/model.pth', map_location=device))
baseline_model.eval()

robust_model = CNN().to(device)
robust_model.load_state_dict(torch.load('../results/model_robust.pth', map_location=device))
robust_model.eval()

print('Both models loaded.')
print(f'Device: {device}')


Both models loaded.
Device: cpu


In [ ]:
# TABLE 1: Attack Success Rate on Baseline Model
# Все атаки при eps=0.03 (FGSM/PGD) и eps=0.10 (GA), N=1000/100, seed=42

import pandas as pd

table1 = {
    'Method':  ['FGSM', 'PGD', 'GA (ours)'],
    'Type':    ['White-box', 'White-box', 'Black-box'],
    'Epsilon': [0.03, 0.03, 0.10],
    'ASR':     [0.5451, 0.7461, 0.3571],
    'Queries': ['N/A (gradient)', 'N/A (gradient)', '1500 (pop=50 × gen=30)'],
}

df1 = pd.DataFrame(table1)
print('Table 1: Attack Success Rate on Baseline CNN (Clean Acc = 83.1%)')
print(df1.to_string(index=False))


Table 1: Attack Success Rate on Baseline CNN (Clean Acc = 83.1%)
 Method        Type  Epsilon     ASR                        Queries
   FGSM   White-box     0.03  0.5451                 N/A (gradient)
    PGD   White-box     0.03  0.7461                 N/A (gradient)
GA (ours)  Black-box     0.10  0.3571  1500 (pop=50 × gen=30)


In [ ]:
# TABLE 2: Defense Effectiveness — Baseline vs Adversarially Trained Model
# Adversarial training: 50/50 mix clean+PGD, eps=0.03, 30 epochs

table2 = {
    'Model':      ['Baseline CNN', 'Robust CNN (adv. trained)'],
    'Clean Acc':  [0.8310, 0.8400],
    'FGSM ASR':   [0.5451, 0.2119],
    'PGD ASR':    [0.7461, 0.2393],
    'GA ASR':     [0.3571, 0.2299],
}

df2 = pd.DataFrame(table2)
print('Table 2: Effect of Adversarial Training on Attack Success Rates')
print(df2.to_string(index=False))
print()
print('Key finding: PGD-based adversarial training reduces gradient attack ASR by ~3x,')
print('but reduces GA ASR by only 1.6x — GA is harder to defend against with this method.')


Table 2: Effect of Adversarial Training on Attack Success Rates
                      Model  Clean Acc  FGSM ASR  PGD ASR  GA ASR
                Baseline CNN     0.8310    0.5451   0.7461  0.3571
  Robust CNN (adv. trained)     0.8400    0.2119   0.2393  0.2299

Key finding: PGD-based adversarial training reduces gradient attack ASR by ~3x,
but reduces GA ASR by only 1.6x — GA is harder to defend against with this method.


In [ ]:
# FIGURE: GA fitness evolution across generations
# Already saved in notebook 03 — just reference here
from PIL import Image
img = Image.open('../results/figures/ga_fitness_evolution.png')
plt.figure(figsize=(8,4))
plt.imshow(img)
plt.axis('off')
plt.title('Figure 3: GA Fitness Evolution')
plt.show()